# Classroom Attendance Prediction: Phase 3 — Feature Engineering & Splitting
## Capstone Project: Classroom Attendance Prediction Using Academic Schedule and Historical Attendance Data

### 📌 Project Objective:
This notebook constructs high-signal, leakage-free predictive features from raw timetable attributes and historical attendance logs. 

---
### 🔒 Target Leakage Safeguards (Strict Audit):
1. **Target Isolation**: Neither `Attendance Percentage` nor `Students Present` is ever passed as an input feature during inference.
2. **Shifted Autoregressive Lags**: The rolling average feature strictly utilizes prior historical lectures:
   $$\text{Rolling\_3\_Avg}_t = \frac{1}{3}\sum_{i=1}^3 \text{Attendance}_{t-i}$$
3. **Chronological Splitting**: $70\%$ Train, $15\%$ Validation, and $15\%$ Test sets are partitioned strictly in temporal sequence to avoid future data leaking into the past.


### 1. Library Imports


In [ ]:
import os
import pandas as pd
import numpy as np

print("Feature engineering modules initialized.")


### 2. Load Cleaned Dataset


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np

def find_data_file(filename="attendance_raw.csv"):
    """
    Auto-discovers datasets and model artifacts in Kaggle input/working directories
    or local relative repository folders.
    """
    ext = os.path.splitext(filename)[1].lower()

    # 1. Search Kaggle input paths
    kaggle_input = "/kaggle/input"
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            if filename in files:
                p = os.path.join(root, filename)
                print(f"[Kaggle Input] Found: {p}")
                return p
            for f in files:
                if ext and f.lower().endswith(ext) and filename.lower().replace(ext, "") in f.lower():
                    p = os.path.join(root, f)
                    print(f"[Kaggle Input] Found matching file: {p}")
                    return p

    # 2. Search Kaggle working directory
    if os.path.exists("/kaggle/working"):
        p = os.path.join("/kaggle/working", filename)
        if os.path.exists(p):
            print(f"[Kaggle Working] Found: {p}")
            return p
        for root, dirs, files in os.walk("/kaggle/working"):
            if filename in files:
                p = os.path.join(root, filename)
                print(f"[Kaggle Working Tree] Found: {p}")
                return p

    # 3. Search local project paths
    local_candidates = [
        os.path.join("data", "processed", filename),
        os.path.join("..", "data", "processed", filename),
        os.path.join("data", "raw", filename),
        os.path.join("..", "data", "raw", filename),
        os.path.join("models", filename),
        os.path.join("..", "models", filename),
        os.path.join("reports", filename),
        os.path.join("..", "reports", filename),
        filename,
        os.path.join("..", filename)
    ]
    for p in local_candidates:
        if os.path.exists(p):
            print(f"[Local Path] Found: {p}")
            return p

    # 4. Search recursively in current working tree
    for root, dirs, files in os.walk("."):
        if filename in files:
            p = os.path.join(root, filename)
            print(f"[Tree Search] Found: {p}")
            return p

    raise FileNotFoundError(f"Could not find '{filename}'.")

def get_output_dir(subfolder=""):
    """Determines writable output directory (/kaggle/working/ or local folder)."""
    if os.path.exists("/kaggle/working"):
        out_dir = os.path.join("/kaggle/working", subfolder) if subfolder else "/kaggle/working"
    else:
        out_dir = os.path.join("..", subfolder) if os.path.exists("..") else (subfolder if subfolder else ".")
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

# Load cleaned dataset
try:
    data_path = find_data_file("attendance_cleaned.csv")
except FileNotFoundError:
    data_path = find_data_file("attendance_raw.csv")

df = pd.read_csv(data_path)
print(f"Loaded attendance records: {len(df)}")


### 3. Feature Engineering Pipeline Implementation
We construct three families of predictive signals:
1. **Academic Calendar Signals** (`Day_of_Semester`, `Week_Number`, `Days_Since_Holiday`, `Week_Before_Exam_Flag`).
2. **Timetable & Timing Signals** (`Start_Hour`, `Time_of_Day`, `Is_Morning`, `Lunch_Timing`, `Is_After_Lunch`, `Daily_Lecture_Sequence`).
3. **Autoregressive Lag Signals** (`Rolling_Prev_3_Avg_Attendance`, `Macro_Subject_Mean_Attendance`, `Macro_Faculty_Mean_Attendance`).


In [ ]:
def engineer_features(df_input, historical_stats=None, is_training=True):
    df = df_input.copy()
    
    # 1. Parse Start Time to float hours
    def parse_hour(time_str):
        try:
            s = str(time_str).strip().lower()
            if ":" in s:
                parts = s.split(":")
                return float(parts[0]) + float(parts[1][:2]) / 60.0
            return float(s)
        except:
            return 9.0
            
    df["Start_Hour"] = df["Start Time"].apply(parse_hour)
    
    # Time of Day
    conditions = [
        (df["Start_Hour"] < 12.0),
        (df["Start_Hour"] >= 12.0) & (df["Start_Hour"] < 16.5),
        (df["Start_Hour"] >= 16.5)
    ]
    df["Time_of_Day"] = np.select(conditions, ["Morning", "Afternoon", "Evening"], default="Morning")
    df["Is_Morning"] = (df["Start_Hour"] < 12.0).astype(int)
    
    # Lunch Timing
    df["Lunch_Timing"] = np.where((df["Start_Hour"] >= 13.0) | (df["Lecture Number"] >= 4), "After Lunch", "Before Lunch")
    df["Is_After_Lunch"] = (df["Lunch_Timing"] == "After Lunch").astype(int)
    
    # 2. Academic Calendar Features
    df["Date_DT"] = pd.to_datetime(df["Date"], errors="coerce")
    min_date = df["Date_DT"].min()
    df["Day_of_Semester"] = (df["Date_DT"] - min_date).dt.days + 1
    df["Week_Number"] = ((df["Day_of_Semester"] - 1) // 7) + 1
    df["Week_Number"] = df["Week_Number"].clip(lower=1, upper=16)
    
    # Holiday Proximity
    df["Days_Since_Holiday"] = np.where(df["Holiday Before/After"] == "Yes", 1, 7)
    df["Week_Before_Exam_Flag"] = np.where(df["Internal Test Week"] == "Yes", 1, 0)
    
    # 3. Daily Cohort Sequence
    df = df.sort_values(by=["Date_DT", "Start_Hour", "Lecture Number"]).reset_index(drop=True)
    df["Daily_Lecture_Sequence"] = df.groupby(["Date", "Branch", "Section"]).cumcount() + 1
    
    # 4. Leakage-Free Shifted Rolling 3 Average
    df["Rolling_Prev_3_Avg_Attendance"] = (
        df.groupby(["Subject", "Branch", "Section"])["Attendance Percentage"]
        .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
    )
    # Impute missing initial lags with cohort historical mean
    global_mean = df["Attendance Percentage"].mean()
    df["Rolling_Prev_3_Avg_Attendance"] = df["Rolling_Prev_3_Avg_Attendance"].fillna(df["Previous Lecture Attendance"]).fillna(global_mean)
    
    # 5. Macro Subject & Faculty Historical Means
    if is_training:
        sub_means = df.groupby("Subject")["Attendance Percentage"].mean().to_dict()
        fac_means = df.groupby("Faculty ID")["Attendance Percentage"].mean().to_dict()
        historical_stats = {"sub_means": sub_means, "fac_means": fac_means, "global_mean": global_mean}
    
    df["Macro_Subject_Mean_Attendance"] = df["Subject"].map(historical_stats["sub_means"]).fillna(historical_stats["global_mean"])
    df["Macro_Faculty_Mean_Attendance"] = df["Faculty ID"].map(historical_stats["fac_means"]).fillna(historical_stats["global_mean"])
    df["Monthly_Avg_Attendance"] = historical_stats["global_mean"]
    
    df = df.drop(columns=["Date_DT"])
    return df, historical_stats

df_feat, hist_stats = engineer_features(df, is_training=True)
print(f"Feature Engineering Complete! Dimensions: {df_feat.shape[0]} rows, {df_feat.shape[1]} columns")
display(df_feat[["Subject", "Lecture Number", "Start_Hour", "Time_of_Day", "Lunch_Timing", "Rolling_Prev_3_Avg_Attendance", "Macro_Subject_Mean_Attendance"]].head(5))


### 4. Chronological Splitting (70% Train, 15% Val, 15% Test)
We partition the dataset chronologically to strictly test models on future lectures.


In [ ]:
n = len(df_feat)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_feat.iloc[:train_end].copy().reset_index(drop=True)
val_df = df_feat.iloc[train_end:val_end].copy().reset_index(drop=True)
test_df = df_feat.iloc[val_end:].copy().reset_index(drop=True)

print(f"=== Chronological Split Breakdown ===")
print(f" - Training Set   : {len(train_df)} rows ({len(train_df)/n*100:.1f}%) | Dates: {train_df['Date'].min()} to {train_df['Date'].max()}")
print(f" - Validation Set : {len(val_df)} rows ({len(val_df)/n*100:.1f}%) | Dates: {val_df['Date'].min()} to {val_df['Date'].max()}")
print(f" - Test Set       : {len(test_df)} rows ({len(test_df)/n*100:.1f}%) | Dates: {test_df['Date'].min()} to {test_df['Date'].max()}")


### 5. Export Engineered Datasets


In [ ]:
out_dir = get_output_dir("data/processed" if not os.path.exists("/kaggle/working") else "")

train_df.to_csv(os.path.join(out_dir, "train_engineered.csv"), index=False)
val_df.to_csv(os.path.join(out_dir, "val_engineered.csv"), index=False)
test_df.to_csv(os.path.join(out_dir, "test_engineered.csv"), index=False)
print(f"[OK] Engineered splits exported successfully to: {out_dir}")


### 6. Phase 3 Summary:
- Constructed academic calendar, timing, and autoregressive lag signals.
- Confirmed zero target leakage.
- Split data chronologically into Train, Validation, and Test partitions.
